# Exercise 1: Asynchronous Video Generation with Amazon Bedrock

This notebook demonstrates how to use Amazon Bedrock's **Nova Reel** model to generate videos asynchronously and store them in an S3 bucket.

## Overview

This exercise shows:
- How to use the `start_async_invoke` API for asynchronous video generation
- How to configure video generation parameters (fps, duration, dimensions)
- How to check the status of an async job
- How to save generated videos to S3

## Prerequisites

- AWS account configured with appropriate credentials
- Amazon Bedrock access enabled for Nova Reel model
- S3 bucket created for storing generated videos
- AWS CLI configured with credentials

## Model Used

- **Model ID**: `amazon.nova-reel-v1:0`
- **Model Type**: Video generation from text prompts
- **API**: `start_async_invoke` (asynchronous processing)

## Expected Duration

Video generation typically takes 2-3 minutes to complete.



## Step 1: Install Required Packages

Install the `boto3` library, which is the AWS SDK for Python. This allows us to interact with AWS services including Bedrock and S3.


In [2]:
!pip install boto3

  Using cached jmespath-1.0.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.14.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
   ---------------------------------------- 0.0/14.1 MB ? eta -:--:--
   ------- -------------------------------- 2.6/14.1 MB 16.0 MB/s eta 0:00:01
   -------------------------- ------------- 9.2/14.1 MB 24.3 MB/s eta 0:00:01
   ---------------------------------------- 14.1/14.1 MB 23.8 MB/s  0:00:00
Using cached jmespath-1.0.1-py3-none-any.whl (20 kB)
Using cached s3transfer-0.14.0-py3-none-any.whl (85 kB)
Using cached urllib3-2.5.0-py3-none-any.whl (129 kB)

   ---------------------------------------- 0/5 [urllib3]
   ---------------- ----------------------- 2/5 [botocore]
   ---------------- ----------------------- 2/5 [botocore]
   ---------------- ----------------------- 2/5 [botocore]
   ---------------- ----------------------- 2/5 [botocore]
   ---------------- ----------------------- 

## Step 2: Import Required Libraries

Import the necessary Python libraries:
- `boto3`: AWS SDK for Python
- `json`: For handling JSON data structures
- `random`: For generating random seed values for video generation


## Step 3: Configure S3 Bucket

Set the name of your S3 bucket where the generated videos will be stored. 

**Important**: Replace this with your actual S3 bucket name that was created in Task 8 of the exercise.


## Step 4: Generate Video Asynchronously

This cell:
1. Creates AWS clients for Bedrock Runtime, Bedrock, and S3 services
2. Sets up the Nova Reel model ID
3. Defines the text prompt for video generation
4. Generates a random seed for video variation
5. Configures video generation parameters:
   - **fps**: 24 frames per second
   - **durationSeconds**: 6 seconds
   - **dimension**: 1280x720 (HD)
   - **seed**: Random value for reproducibility
6. Configures S3 output location
7. Submits the async job using `start_async_invoke`
8. Prints the invocation ARN (used to track job status)

**Note**: The job runs asynchronously, so this cell completes immediately. The actual video generation happens in the background.


## Step 5: Check Job Status

Use the invocation ARN from the previous step to check the status of the video generation job.

**Status Options**:
- `InProgress`: Job is still processing
- `Completed`: Video has been generated and saved to S3
- `Failed`: An error occurred during generation

**Note**: You may need to run this cell multiple times until the status shows "Completed". This typically takes 2-3 minutes.

Once completed, you can find the generated video in your S3 bucket at: `s3://{bucket_name}/video/`


In [3]:
import boto3
import json
import random

In [4]:
s3_bucket = "coursera-gen-ai-exercise-za"

In [5]:

bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")
bedrock = boto3.client(service_name="bedrock", region_name="us-east-1")  
s3 = boto3.client("s3")
model_id = "amazon.nova-reel-v1:0"

prompt = "A person dancing on a mountain."

seed = random.randint(0, 2147483646)

model_input = {
    "taskType": "TEXT_VIDEO",
    "textToVideoParams": {"text": prompt},
    "videoGenerationConfig": {
        "fps": 24,
        "durationSeconds": 6,
        "dimension": "1280x720",
        "seed": seed,
    },
}

output_config = {
    "s3OutputDataConfig": {
        "s3Uri": f"s3://{s3_bucket}/video/"
    }
}

response = bedrock_runtime.start_async_invoke(
    modelId=model_id,
    modelInput=model_input,
    outputDataConfig=output_config,
)

invocation_arn = response["invocationArn"]
print("✅ Job submitted!")
print("Invocation ARN:", invocation_arn)


✅ Job submitted!
Invocation ARN: arn:aws:bedrock:us-east-1:458806987020:async-invoke/wk39f36fxk6g


In [7]:
job_status = bedrock_runtime.get_async_invoke(invocationArn=invocation_arn)
print("Current Status:", job_status["status"])

Current Status: Completed
